In [3]:
from datasets import load_dataset
import random
from tqdm import tqdm
import os
import json
import hashlib
import pandas as pd
from collections import defaultdict

In [4]:
def text_to_id(text):
    """Return a deterministic ID for the given text."""
    # Normalize whitespace etc. to avoid accidental differences
    return hashlib.md5(text.encode("utf-8")).hexdigest()

In [11]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [12]:
num_bad_docs = 2

In [13]:
random.seed(42) 

skip_count = 0
train_data = []
for row in tqdm(ds['train'], total=len(ds['train'])):
    query = row['question']
    good_docs = row['supporting_facts']
    good_doc_titles = good_docs["title"]
    all_docs = row['context']
    all_doc_titles = all_docs["title"]
    # print(len(all_doc_titles), len(good_doc_titles))
    good_doc_ids = [i for i, title in enumerate(all_doc_titles) if title in good_doc_titles]
    if len(all_doc_titles) >= len(good_doc_ids) + num_bad_docs:
        good_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in good_doc_ids]
        for gd in good_doc_texts:
            bad_doc_ids = random.choices([i for i in range(len(all_doc_titles)) if i not in good_doc_ids], k=num_bad_docs)
            bad_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in bad_doc_ids]
            for bd in bad_doc_texts:
                train_data.append({
                    "query": query,
                    "good_doc": gd,
                    "bad_doc": bd
                })
    else:
        skip_count += 1
    # print(train_data)
    # break
print(f"Skipped {skip_count} samples due to insufficient bad documents.")

100%|██████████| 90447/90447 [00:14<00:00, 6324.97it/s]

Skipped 418 samples due to insufficient bad documents.


In [14]:
random.seed(42) 

skip_count = 0
validation_data = []
for row in tqdm(ds['validation'], total=len(ds['validation'])):
    query = row['question']
    good_docs = row['supporting_facts']
    good_doc_titles = good_docs["title"]
    all_docs = row['context']
    all_doc_titles = all_docs["title"]
    # print(len(all_doc_titles), len(good_doc_titles))
    good_doc_ids = [i for i, title in enumerate(all_doc_titles) if title in good_doc_titles]
    if len(all_doc_titles) >= len(good_doc_ids) + num_bad_docs:
        good_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in good_doc_ids]
        for gd in good_doc_texts:
            bad_doc_ids = random.choices([i for i in range(len(all_doc_titles)) if i not in good_doc_ids], k=num_bad_docs)
            bad_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in bad_doc_ids]
            for bd in bad_doc_texts:
                validation_data.append({
                    "query": query,
                    "good_doc": gd,
                    "bad_doc": bd
                })
    else:
        skip_count += 1
    # print(validation_data)
    # break
print(f"Skipped {skip_count} samples due to insufficient bad documents.")

100%|██████████| 7405/7405 [00:01<00:00, 5852.61it/s]

Skipped 28 samples due to insufficient bad documents.


In [15]:
len(train_data), len(validation_data)

(360116, 29508)

In [16]:
os.makedirs("data", exist_ok=True)
with open("data/train_data.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")
with open("data/validation_data.jsonl", "w") as f:
    for item in validation_data:
        f.write(json.dumps(item) + "\n")

In [17]:
os.makedirs("data/docs", exist_ok=True)
doc_ids = set()
with open("data/docs/doc_data.jsonl", "w") as f:
    for row in tqdm(ds['train'], total=len(ds['train'])):
        query = row['question']
        all_docs = row['context']
        query_id = text_to_id(query)
        if query_id not in doc_ids:
            doc_ids.add(query_id)
            f.write(json.dumps({
                "id": query_id,
                "contents": query
            }) + "\n")
        for i in range(len(all_docs["title"])):
            doc_text = " ".join([s.strip() for s in all_docs["sentences"][i]])
            doc_id = text_to_id(doc_text)
            if doc_id not in doc_ids:
                doc_ids.add(doc_id)
                f.write(json.dumps({
                    "id": doc_id,
                    "contents": doc_text
                }) + "\n")
    for row in tqdm(ds['validation'], total=len(ds['validation'])):
        query = row['question']
        all_docs = row['context']
        query_id = text_to_id(query)
        if query_id not in doc_ids:
            doc_ids.add(query_id)
            f.write(json.dumps({
                "id": query_id,
                "contents": query
            }) + "\n")
        for i in range(len(all_docs["title"])):
            doc_text = " ".join([s.strip() for s in all_docs["sentences"][i]])
            doc_id = text_to_id(doc_text)
            if doc_id not in doc_ids:
                doc_ids.add(doc_id)
                f.write(json.dumps({
                    "id": doc_id,
                    "contents": doc_text
                }) + "\n")
print(f"Wrote {len(doc_ids)} unique documents.")

100%|██████████| 7405/7405 [00:01<00:00, 4432.42it/s]

Wrote 606960 unique documents.


In [5]:
qrel_path = "simple_english/en2simple.rel"
queries_path = "simple_english/wiki_en.queries"
docs_path = "simple_english/wiki_simple.documents"

In [6]:
qrel_df = pd.read_csv(qrel_path, names=["query_id", "doc_id", "relevance"], delimiter="\t")
qrel_df.head()

,query_id,doc_id,relevance
0,12,4807,2
1,12,4080,1
2,12,60610,1
3,12,798,1
4,12,12446,1


In [7]:
qrel_dict = defaultdict(list)
for qrel in qrel_df.itertuples():
    qrel_dict[qrel.query_id].append(qrel.doc_id)
with open("data/qrel_dict.json", "w") as f:
    json.dump(qrel_dict, f, indent=4)

In [8]:
doc_df = pd.read_csv(docs_path, names=["doc_id", "title", "text"], delimiter="\t")
doc_df.head()

,doc_id,title,text
0,1,April,"april is the fourth month of the year , and c..."
1,2,August,august is the eighth month of the year in the...
2,6,Art,art is a creative activity by people . the ar...
3,8,A,a is the first letter of the english alphabet...
4,9,Air,air is the earth 's atmosphere . air around u...


In [9]:
queries_df = pd.read_csv(queries_path, names=["query_id", "title", "question"], delimiter="\t")
queries_df.head()

,query_id,title,question
0,12,Anarchism,is a political philosophy that advocates self-...
1,25,Autism,is a neurodevelopmental disorder characterized...
2,39,Albedo,() is a measure for reflectance or optical bri...
3,290,A,"(named , plural ""as"", ""a's"", ""a""s, ""a's"" or ""a..."
4,303,Alabama,() is a state in the southeastern region of th...


In [10]:
# Sample 5000 queries with most relevant documents using qrel_dict
sorted_query_ids = sorted(qrel_dict.keys(), key=lambda qid: len(qrel_dict[qid]), reverse=True)
sampled_query_ids = [id for id in sorted_query_ids if len(qrel_dict[id]) >= 10]
sampled_query_df = queries_df[queries_df['query_id'].isin(sampled_query_ids)]
len(sampled_query_df)

2205

In [11]:
sampled_query_df.to_json("data/test_queries.jsonl", orient="records", lines=True)

In [12]:
for row in sampled_query_df.itertuples():
    q_id = row.query_id
    assert q_id in qrel_dict
    for bd_id in qrel_dict[q_id]:
        assert bd_id in doc_df['doc_id'].values

In [13]:
all_good_docs = set()
for q_id in sampled_query_df['query_id'].values:
    for gd_id in qrel_dict[q_id]:
        all_good_docs.add(gd_id)
len(all_good_docs)
doc_df_filtered = doc_df[doc_df['doc_id'].isin(all_good_docs)]
doc_df_filtered.to_json("data/test_documents.jsonl", orient="records", lines=True)

In [14]:
os.makedirs("data/docs_test", exist_ok=True)
with open("data/docs_test/doc_data.jsonl", "w") as f:
    doc_ids = set()
    for _, row in sampled_query_df.iterrows():
        query_text = row['question']
        query_id = text_to_id(query_text)
        if query_id not in doc_ids:
            doc_ids.add(query_id)
            f.write(json.dumps({
                "id": query_id,
                "contents": query_text
            }) + "\n")
    for _, row in doc_df.iterrows():
        doc_text = row['text']
        doc_id = text_to_id(doc_text)
        if doc_id not in doc_ids:
            doc_ids.add(doc_id)
            f.write(json.dumps({
                "id": doc_id,
                "contents": doc_text
            }) + "\n")